# Modeling: 배터리 수명 예측 모델 개발 및 평가

설계된 Feature Sets를 바탕으로 점진적 비교 실험(Ablation)을 거쳐 최적의 Feature Set을 선정하고, 기준 모델(Ridge 회귀 파이프라인) 학습 및 공식 리포트 생성

## 0. 필수 라이브러리 및 경로 설정

In [21]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 경로 설정
CURRENT_DIR = Path.cwd().resolve()
ROOT = CURRENT_DIR.parent if CURRENT_DIR.name in ('notebooks', 'test') else CURRENT_DIR
DATA_DIR = ROOT / 'results' / 'features'
OUTPUT_DIR = ROOT / 'results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'데이터 경로: {DATA_DIR}')
print(f'산출물 저장 경로: {OUTPUT_DIR}')

데이터 경로: /Users/yujin.je/Workspace/ds-mini/results/features
산출물 저장 경로: /Users/yujin.je/Workspace/ds-mini/results


## 1. 데이터 로드 및 평가 지표 정의
- **데이터 로드**: 02_feature_engineering에서 분할 저장된 Raw 데이터셋 및 `feature_sets.json` 로드
- **주요 평가 지표**: *Nature Energy* 원논문 벤치마크 기준인 MAPE(목표 9.1%) 산출 함수 정의
- **Valid 세트에 Hold-out 검증 방식을 채택한 이유**:
  - 셀 단위 독립성: 배터리 데이터는 셀 단위로 독립적이며 각 셀이 서로 다른 충전 프로토콜로 실험됨
  - 누수 차단: 무작위 교차 검증 적용 시 동일 프로토콜 셀이 Train과 Valid에 나뉘어 들어가 데이터 누수 위험 잔존
  - 일반화 평가 적합성: 충전 프로토콜 기준 분할을 통해 프로토콜 단위의 완전한 분리를 보장하고 배치 간 일반화를 엄밀히 평가하기에 더 적합

In [22]:
# 데이터셋 로드
train_df = pd.read_csv(DATA_DIR / 'train_features_raw.csv')
val_df = pd.read_csv(DATA_DIR / 'valid_features_raw.csv')
test_b2 = pd.read_csv(DATA_DIR / 'test_b2_features_raw.csv')
test_b3 = pd.read_csv(DATA_DIR / 'test_b3_features_raw.csv')

with open(DATA_DIR / 'feature_sets.json', 'r', encoding='utf-8') as f:
    FEATURE_SETS = json.load(f)

# 타겟 정의
y_train = train_df['cycle_life'].values
y_val = val_df['cycle_life'].values
y_b2 = test_b2['cycle_life'].values
y_b3 = test_b3['cycle_life'].values

# MAPE 산출 함수
def calculate_mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print(f'Train: {len(train_df)}개 | Valid: {len(val_df)}개 | Test B2: {len(test_b2)}개 | Test B3: {len(test_b3)}개')
print(f'Feature Sets: {list(FEATURE_SETS.keys())}')

Train: 35개 | Valid: 11개 | Test B2: 39개 | Test B3: 44개
Feature Sets: ['A', 'B', 'C', 'D', 'ALT_delta_q_min', 'E_mean_QD']


## 2. Feature Sets 점진적 비교 실험 (Ablation)
- **실험 목적**: 피처 조합에 따른 교차 검증 오차(Train CV MAPE 및 CV Std)와 검증 오차(Valid MAPE) 비교
- **교차 검증**: 데이터 누수 방지를 위해 Train 세트 내부 `charging_policy` 기준 `GroupKFold(n_splits=4)` 교차 검증 적용
- **파이프라인 누수 방지**: 결측 대치 및 표준화 스케일러는 각 Fold 내부의 학습 데이터에만 `fit` 수행
- **테스트 세트 누수 배제**: 외부 테스트셋(Batch 2, Batch 3)은 모델 및 피처 선정 단계에서 철저히 배제

In [23]:
def evaluate_pipeline_cv(model, X, y, groups, n_splits=4):
    gkf = GroupKFold(n_splits=n_splits)
    scores = []
    for tr_idx, val_idx in gkf.split(X, y, groups=groups):
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('model', model)
        ])
        pipe.fit(X.iloc[tr_idx], y[tr_idx])
        pred = pipe.predict(X.iloc[val_idx])
        scores.append(calculate_mape(y[val_idx], pred))
    return np.mean(scores), np.std(scores)

ablation_results = []
for set_name, cols in FEATURE_SETS.items():
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=1.0, random_state=42))
    ])
    
    cv_mean, cv_std = evaluate_pipeline_cv(Ridge(alpha=1.0, random_state=42), train_df[cols], y_train, train_df['charging_policy'])
    pipe.fit(train_df[cols], y_train)
    val_pred = pipe.predict(val_df[cols])
    
    ablation_results.append({
        'Feature Set': set_name,
        '피처 수': len(cols),
        'Train CV MAPE (%)': round(cv_mean, 2),
        'CV Std': round(cv_std, 2),
        'Valid MAPE (%)': round(calculate_mape(y_val, val_pred), 2),
    })

ablation_df = pd.DataFrame(ablation_results)
display(ablation_df)

Feature Set,피처 수,Train CV MAPE (%),CV Std,Valid MAPE (%)
A,1,7.08,0.98,12.72
B,4,8.73,1.38,12.40
C,7,8.27,0.74,14.90
D,8,9.66,2.36,50.36
ALT_delta_q_min,1,8.89,1.98,9.41
E_mean_QD,9,8.99,3.44,42.48


### Feature Set 비교 분석 및 후보 선정
- **Set A 및 ALT_delta_q_min 우선 후보 선정**:
  - Set A(`log_delta_q_var`): Train CV MAPE 7.08% (Std 0.98)로 폴드 간 가장 안정적이고 우수한 예측 성능 기록
  - ALT_delta_q_min(`delta_q_min`): Valid MAPE 9.41%로 내부 검증 세트에서 가장 낮은 오차 달성
  - 단일 피처 기반 신호가 복잡한 다변수 세트 대비 과적합 없이 일관된 성능 유지
- **보조 Feature Sets(Set C, D, E) 탈락 사유**:
  - 충전시간, 저항, 온도 등 물리 보조 신호 및 용량 지표 추가 시 Valid MAPE가 14.9%~50.4%로 급증하며 폴드 간 변동성(CV Std)도 확대
  - 소표본 환경(N=35)에서의 과적합 확인에 따라 단일 핵심 물리 신호(Set A, ALT) 중심으로 후보군 압축

## 3. 최종 모델 학습 및 교차 검증
- **선정된 Feature Set**: Set A(`log_delta_q_var`) 기반 최종 Ridge 회귀 파이프라인 학습 및 전체 검증

In [24]:
selected_features = FEATURE_SETS['A']

# 최종 모델 파이프라인 구축
ridge_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0, random_state=42))
])

# 학습
ridge_pipeline.fit(train_df[selected_features], y_train)

# 예측
train_cv_mape, train_cv_std = evaluate_pipeline_cv(Ridge(alpha=1.0, random_state=42), train_df[selected_features], y_train, train_df['charging_policy'])
valid_pred = ridge_pipeline.predict(val_df[selected_features])
b2_pred = ridge_pipeline.predict(test_b2[selected_features])
b3_pred = ridge_pipeline.predict(test_b3[selected_features])

valid_mape = calculate_mape(y_val, valid_pred)
test_b2_mape = calculate_mape(y_b2, b2_pred)
test_b3_mape = calculate_mape(y_b3, b3_pred)

print(f'Train CV MAPE: {train_cv_mape:.2f}% (Std: {train_cv_std:.2f})')
print(f'Valid MAPE: {valid_mape:.2f}%')
print(f'Test B2 MAPE: {test_b2_mape:.2f}%')
print(f'Test B3 MAPE: {test_b3_mape:.2f}%')

Train CV MAPE: 7.08% (Std: 0.98)
Valid MAPE: 12.72%
Test B2 MAPE: 25.85%
Test B3 MAPE: 12.16%


## 4. 최종 모델 성능 리포팅
- `pjt-info.md` 표준 양식에 맞추어 Train, Valid, Test B2, Test B3 성능 및 Gap 테이블 생성 및 CSV 저장
- **리포트 항목 기준**:
  - Train (Batch 1 CV): Batch 1 내부 정책 기준 `GroupKFold(n_splits=4)` 교차 검증 평균 성능
  - Valid (Batch 1 Hold-out): 프로토콜 중복 누수를 차단한 고정 검증 세트 성능
  - Test (Batch 2 / Batch 3): 미학습 외부 배치 데이터셋 대상 최종 일반화 평가 성능

In [25]:
TARGET_MAPE = 9.1

report_rows = [
    {'구분': 'Train (Batch 1 CV)', 'MAPE (%)': f'{train_cv_mape:.2f}', '비고': ''},
    {'구분': 'Valid (Batch 1 Hold-out)', 'MAPE (%)': f'{valid_mape:.2f}', '비고': ''},
    {'구분': 'Test (Batch 2)', 'MAPE (%)': f'{test_b2_mape:.2f}', '비고': ''},
    {'구분': '  Gap (Train-Valid)', 'MAPE (%)': f'{valid_mape - train_cv_mape:+.2f}', '비고': '(+) : 과적합 의심'},
    {'구분': '  Gap (Valid-Test)', 'MAPE (%)': f'{test_b2_mape - valid_mape:+.2f}', '비고': '(+) : 배치간 일반화 저하 의심'},
    {'구분': '  Gap (Target-Test)', 'MAPE (%)': f'{test_b2_mape - TARGET_MAPE:+.2f}', '비고': 'Target : 원논문 9.1%'},
    {'구분': 'Test (Batch 3)', 'MAPE (%)': f'{test_b3_mape:.2f}', '비고': ''},
    {'구분': '  Gap (Batch2-Batch3)', 'MAPE (%)': f'{test_b3_mape - test_b2_mape:+.2f}', '비고': '(+) : Batch 3 성능 저하'},
    {'구분': '  Gap (Target-Test)', 'MAPE (%)': f'{test_b3_mape - TARGET_MAPE:+.2f}', '비고': 'Batch 3 기준, 원논문 9.1% 비교'},
]

report_df = pd.DataFrame(report_rows)
display(report_df)

report_df.to_csv(OUTPUT_DIR / 'model_performance.csv', index=False, encoding='utf-8-sig')
print(f'성능 리포트 저장 완료: {OUTPUT_DIR / "model_performance.csv"}')

,구분,MAPE (%),비고
0,Train (Batch 1 CV),7.08,
1,Valid (Batch 1 Hold-out),12.72,
2,Test (Batch 2),25.85,
3,Gap (Train-Valid),+5.64,(+) : 과적합 의심
4,Gap (Valid-Test),+13.12,(+) : 배치간 일반화 저하 의심
5,Gap (Target-Test),+16.75,Target : 원논문 9.1%
6,Test (Batch 3),12.16,
7,Gap (Batch2-Batch3),-13.69,(+) : Batch 3 성능 저하
8,Gap (Target-Test),+3.06,"Batch 3 기준, 원논문 9.1% 비교"


성능 리포트 저장 완료: /Users/yujin.je/Workspace/ds-mini/results/model_performance.csv
